In [1]:
!pip install transformers datasets seqeval scikit-learn jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=bf48df9e53ec342a47f6040638645fa057f43e9eb3d18983486cbd96e44b5853
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import json
import jsonlines
import torch
import spacy
import numpy as np

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score


# -------- MODEL --------
MODEL_NAME = "distilbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

nlp = spacy.load("en_core_web_sm")

labels = {
    "TOPIC": "main subject or research topic",
    "METHOD": "method or technique used to perform a task",
    "CONCEPT": "general concept or idea in computer science",
    "ALGORITHM": "step by step computational algorithm",
    "OTHER": "other technical term"
}


# -------- FAST BATCH EMBEDDING --------
def embed_batch(texts):

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()


# -------- LABEL EMBEDDINGS --------
label_names = list(labels.keys())
label_desc = list(labels.values())

label_embeddings = embed_batch(label_desc)


# -------- CANDIDATE EXTRACTION --------
def extract_candidates(text):

    doc = nlp(text)

    candidates = set()

    for chunk in doc.noun_chunks:
        candidates.add(chunk.text.strip())

    return list(candidates)


# -------- PREDICT LABELS --------
def predict_entities(text):

    candidates = extract_candidates(text)

    if len(candidates) == 0:
        return []

    entity_embeddings = embed_batch(candidates)

    sims = cosine_similarity(entity_embeddings, label_embeddings)

    predictions = []

    for i, entity in enumerate(candidates):

        best_label_idx = np.argmax(sims[i])
        best_label = label_names[best_label_idx]

        predictions.append({
            "entity": entity,
            "label": best_label
        })

    return predictions


# -------- LOAD DATA --------
data = []

with jsonlines.open("OS-cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)


# -------- EVALUATION --------
y_true = []
y_pred = []
partial_scores = []


def compute_accuracy(y_true, y_pred):

    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)


def compute_partial_accuracy(gold_entities, pred_entities):

    match = 0

    for g in gold_entities:

        g_text = g["entity"].lower()

        for p in pred_entities:

            p_text = p["entity"].lower()

            if g_text in p_text or p_text in g_text:
                match += 1
                break

    return match / len(gold_entities)


for sample in data:

    text = sample["input"]
    gold = json.loads(sample["output"])

    pred = predict_entities(text)

    gold_dict = {g["entity"]: g["label"] for g in gold}
    pred_dict = {p["entity"]: p["label"] for p in pred}

    for ent in gold_dict:

        y_true.append(gold_dict[ent])

        if ent in pred_dict:
            y_pred.append(pred_dict[ent])
        else:
            y_pred.append("OTHER")

    p_score = compute_partial_accuracy(gold, pred)
    partial_scores.append(p_score)


# -------- METRICS --------
micro_precision = precision_score(y_true, y_pred, average="micro")
micro_recall = recall_score(y_true, y_pred, average="micro")
micro_f1 = f1_score(y_true, y_pred, average="micro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")
macro_f1 = f1_score(y_true, y_pred, average="macro")

accuracy = compute_accuracy(y_true, y_pred)
partial_accuracy = sum(partial_scores) / len(partial_scores)

print("\n===== Evaluation Results =====")

print("Accuracy:", round(accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))

print("\nMicro Precision:", round(micro_precision,4))
print("Micro Recall:", round(micro_recall,4))
print("Micro F1:", round(micro_f1,4))

print("\nMacro Precision:", round(macro_precision,4))
print("Macro Recall:", round(macro_recall,4))
print("Macro F1:", round(macro_f1,4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== Evaluation Results =====
Accuracy: 0.1462
Partial Accuracy: 0.9335

Micro Precision: 0.1462
Micro Recall: 0.1462
Micro F1: 0.1462

Macro Precision: 0.0727
Macro Recall: 0.1463
Macro F1: 0.0415


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
